# 17 — Does a third tied tag help? (DR-60)

Compare the approved two-tag configuration with a new policy: allow a third tag only when it is equally central and supported by the title. An uncertain guess is not a tie.

We use the same **54 accepted development events** and DR-61 reference answers for both policies. Three unresolved events are excluded. Reference answers are never shortened to fit the model. This is a targeted, repeatedly reviewed sample, not a fresh final test.

**Private inputs required.** Public readers can use `docs/results/tag-policy.md`. Event titles, answers, raw responses and tie explanations stay in ignored `reports/`. Default execution replays saved calls; set `NOTEBOOK17_RUN_LIVE=true` in local `.env` to fill missing calls. Search stays off. No reference answers or review notes are sent.

## 1. Fix the comparison before calling models

Reuse saved v1 predictions and request v2 from Qwen3.6 on Hetzner and Gemini 3.8 Flash. The rubric, titles and generation settings stay unchanged; only the output instruction changes. The tie flag and explanation are part of that change. Different inference dates and provider variation mean this is a practical policy comparison, not proof that every change was caused by the tag limit alone.

In [ ]:
import json, os, sys, copy
from pathlib import Path
from statistics import median
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path[:0] = [str(root), str(root / 'src')]
from notebooks.tools import verify_sha256, show_table, percent
from notebooks.tools.hashes import fingerprint
from notebooks.tools.hosted import chat_request
from notebooks.tools.gemini_evidence import request
from notebooks.tools.costs import BudgetGuard
from notebooks.tools.tag_policy import parse_tied_tags
from demo_logic.evaluation import multilabel_metrics
from demo_logic.taxonomy import topic_tags
from demo_logic.notebook_support import save_private_json
load_dotenv(root / '.env', override=False)
RUN_LIVE = os.getenv('NOTEBOOK17_RUN_LIVE', 'false').lower() == 'true'
INPUTS = {'reports/error-analysis/2026-10-05/approved-reference-v1.json': 'b508d0a3b9b803f7d018816c753e1673860c1a297acd8733607bf6c1f67fe6fe', 'reports/gemini-broader/v1/development-results.json': '59849e7e463a30553454c0ff16ed9fc50f0e4e13c6d5d1f134f9de14e008490b', 'notebooks/prompts/practical-tagging-v1.json': '9b13b3125ab861118cad76cc6e90376fdb2eb3c536d343a12187b5e973538f8a', 'notebooks/prompts/practical-tagging-v2.json': 'b20bbc3418aadfdaa56dfe58c0db44949dce02a0242a916af9fb0d94c8afcf1a', 'docs/tagging-rubric.md': '0127f65668271a6bc32ec6de32eb68c146228696ba6ecb69fbaf1a46cdba70bd'}
missing = [p for p in INPUTS if not (root / p).is_file()]
if missing:
    raise FileNotFoundError('Private inputs missing: ' + ', '.join(missing) + '. Public summary: docs/results/tag-policy.md')
for path, expected in INPUTS.items():
    verify_sha256(root / path, expected)
reference = json.loads((root / list(INPUTS)[0]).read_text())
baseline = json.loads((root / list(INPUTS)[1]).read_text())
v1 = json.loads((root / list(INPUTS)[2]).read_text())
v2 = json.loads((root / list(INPUTS)[3]).read_text())
records = {x['external_id']: x for x in reference['records'] if x['score_eligible']}
ids = sorted(records)
assert len(ids) == 54 and reference['score_ready']
tags = sorted(topic_tags('v2'))
folder = root / 'reports/tag-policy/v1'
contract = dict(version='dr60-v1', ids=ids, inputs=INPUTS, search=False, final_test_used=False,
    baseline='Saved v1 calls; new v2 calls', paid_run_limit_usd=3,
    spending_observation={'date':'2026-10-05','reported_monthly_eur':1.86,'source':'User; not added to USD ledger estimates'})
save_private_json(folder / 'contract.json', contract)
old = {c['model']: {x['external_id']: x for x in baseline['answers'][c['model']]} for c in v1['candidates']}
for c in v1['candidates']:
    for i in ids:
        expected = copy.deepcopy(c['payload_template'])
        if 'messages' in expected:
            expected['messages'][1]['content'] = json.dumps({'title': records[i]['title']}, ensure_ascii=False)
        else:
            expected['input'] = json.dumps({'title': records[i]['title']}, ensure_ascii=False)
        # Serialization whitespace is irrelevant after decoding the title object.
        saved = copy.deepcopy(old[c['model']][i]['payload'])
        if 'messages' in expected:
            assert json.loads(saved['messages'][1]['content']) == json.loads(expected['messages'][1]['content'])
            saved['messages'][1]['content'] = expected['messages'][1]['content']
        else:
            assert json.loads(saved['input']) == json.loads(expected['input'])
            saved['input'] = expected['input']
        assert saved == expected, 'Baseline settings differ from approved v1'
print('54 fixed events; two models; at most 54 new paid calls and 54 free calls.')


## 2. Run or replay the new policy

The ordinary answer has zero to two tags, `tie=false` and an empty reason. Three tags require `tie=true` and a short explanation. The parser checks this structure; it cannot prove that the explanation is correct. We inspect the third-tag cases below.

Each Gemini attempt reserves money before it runs. This experiment has a **$3 conservative ceiling** inside the existing $50 capstone budget. Estimates are not the invoice. Hetzner remains the confirmed free account. Stop on a provider failure and resume later; successful responses are cached.

In [ ]:
guard = BudgetGuard(root / 'reports/budget/capstone.sqlite', prior_total_microusd=0,
    prior_research_microusd=0, run_limits={'dr60-v1':3_000_000})
answers = {}
for c in v2['candidates']:
    model = c['model']; answers[model] = []
    for position, i in enumerate(ids, 1):
        payload = copy.deepcopy(c['payload_template'])
        title_json = json.dumps({'title':records[i]['title']}, ensure_ascii=False)
        if 'messages' in payload:
            payload['messages'][1]['content'] = title_json
            result = chat_request({**c, 'key_env':'HETZNER_API_KEY'}, payload, tags=tags,
                cache_dir=folder / 'qwen/cache', allow_live=RUN_LIVE,
                api_key=os.getenv('HETZNER_API_KEY'), max_retry_wait_seconds=0,
                content_parser=parse_tied_tags)
        else:
            payload['input'] = title_json
            result = request(payload, event_id=i, folder=folder / 'gemini', guard=guard,
                key=os.getenv('GEMINI_API_KEY'), tags=tags, live=RUN_LIVE,
                run_id='dr60-v1', research_budget=False, content_parser=parse_tied_tags)
        # Replay state is not part of the immutable experiment result.
        stable_result = {k:v for k,v in result.items() if k != 'cached'}
        answers[model].append({**stable_result, 'external_id':i})
        if result['status'] != 'ok':
            raise RuntimeError(f"{model}: {result['status']}; progress is saved. Resolve the failure before resuming.")
        if position % 10 == 0 or position == len(ids):
            print(model, position, '/', len(ids), flush=True)


## 3. Compare quality and the third tags

Precision asks how many suggested tags match the reference. Recall asks how many reference tags were found. F1 balances both. Coverage counts events receiving a suggestion. The same 54 events and complete reference answers are used in every row.

We also truncate each **new** answer to its first two tags. This diagnostic isolates the added third tags within the same response. It is not another model run. It helps separate third-tag gains from changes to the first two suggestions.

In [ ]:
truth = [records[i]['reference_tags'] for i in ids]
metrics, third_cases, costs = {}, [], {}
for model, rows in answers.items():
    metrics[model] = {}
    for policy, predictions in [('v1 saved', [old[model][i]['predicted'] for i in ids]),
        ('v2 first two', [x['predicted'][:2] for x in rows]), ('v2 full', [x['predicted'] for x in rows])]:
        metrics[model][policy] = multilabel_metrics(truth, predictions, tags)
    for row in rows:
        if len(row['predicted']) == 3:
            body = row['response']
            if 'choices' in body: text = body['choices'][0]['message']['content']
            else: text = '\n'.join(c.get('text','') for s in body.get('steps',[]) if s.get('type')=='model_output' for c in s.get('content',[]) if c.get('type')=='text')
            text = text.strip()
            if text.startswith('```'): text = text.split('\n',1)[1].rsplit('```',1)[0].strip()
            obj = json.loads(text)
            third_cases.append(dict(model=model, external_id=row['external_id'], title=records[row['external_id']]['title'],
                predicted=row['predicted'], reference=records[row['external_id']]['reference_tags'],
                third_matches=row['predicted'][2] in records[row['external_id']]['reference_tags'], reason=obj['tie_reason']))
    costs[model] = dict(median_original_seconds=median(x['duration_seconds'] for x in rows),
        paid_upper_usd=sum(x.get('gross_upper_microusd',0) or x.get('maximum_microusd',0) for x in rows)/1e6,
        note='Conservative upper estimate, not invoice; Hetzner confirmed free. Replay retains original latency.')
show_table(['Model','Policy','Precision','Recall','F1','Coverage'],
    [[m,p,*[percent(v[k])for k in ['micro_precision','micro_recall','micro_f1','coverage']]] for m, ps in metrics.items() for p,v in ps.items()])
show_table(['Model','Third-tag cases','Third matches reference'],
    [[m,sum(x['model']==m for x in third_cases),sum(x['model']==m and x['third_matches'] for x in third_cases)] for m in answers])
show_table(['Model','Title','Suggested','Reference','Third matches','Tie reason'],
    [[x['model'],x['title'],', '.join(x['predicted']),', '.join(x['reference']),x['third_matches'],x['reason']] for x in third_cases])
save_private_json(folder / 'results.json', dict(contract=contract, answers=answers, metrics=metrics, third_cases=third_cases, costs=costs))


## 4. Visual summary and review

Review the quality table and the third-tag explanations. Does the title support three equally central topics, or did the model add a weaker topic? Agreement with this reference is useful evidence, not an independent second review.

The public summary records the recommendation after the completed run. No production policy changes here. DR-58 should use a versioned, reviewed output policy.

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
labels = []; values = []
for model, ps in metrics.items():
    for policy in ['v1 saved','v2 full']:
        labels.append(('Qwen3.6' if model.startswith('Qwen') else 'Gemini 3.8 Flash') + '\n' + policy)
        values.append(100 * ps[policy]['micro_f1'])
ax.bar(labels,values,color=['#64748b','#10b981','#64748b','#10b981'])
ax.set(ylabel='Micro F1 (%)',ylim=(0,100),title='Same 54 reviewed development events')
for index,value in enumerate(values): ax.text(index,value+1,f'{value:.1f}',ha='center')
fig.tight_layout(); display(fig); plt.close(fig)
show_table(['Model','Median seconds','Upper USD estimate'],[[m,round(c['median_original_seconds'],2),round(c['paid_upper_usd'],4)]for m,c in costs.items()])


## 5. What we learned

**The new policy did not improve overall F1.** Qwen moves from 74.9% to 73.7%; Gemini 3.8 Flash moves from 73.4% to 71.2%. Qwen gains recall but loses precision. Gemini keeps the same recall and loses precision. Coverage is unchanged for both.

Qwen returned three tags twice, with one third tag matching the reference. Gemini did so four times, with two matches. Comparing each new answer with its first two tags shows that the third tags add a small F1 gain within v2. The lower overall result also involves changed first/second suggestions; we cannot blame it entirely on the extra tag.

**Recommendation:** retain v1 as the default for DR-58, pending review. Keep v2 available if the reviewer explicitly prefers the three-tag trade-off. This does not restrict reviewer reference answers to two tags or change production. Neither new result reaches every target.

**Review task:** read section 3 and the five unique events in private `reports/tag-policy/v1/review.md`. Confirm a trial policy. Disagreements can still reflect reference/rubric problems; no approved answers were changed to favor either model.

All 108 new requests completed (54 per model). Hetzner cost $0 on the confirmed free account; Gemini's recorded conservative upper estimate is $0.5994, not the invoice. The earlier €1.86 monthly dashboard figure was reported before this run. Median request times were about 2.02 seconds for Qwen and 3.68 seconds for Gemini. Replaying the notebook makes no calls by default.
